In [ ]:
# Edit only attached Kaggle Input paths and reduce batch sizes only for OOM.
from pathlib import Path

PUBLIC_LEGALIR_PATH = Path("/kaggle/input/datasets/mduy2911/legaluit/LegalIR/public-official.json")
CORPUS_PATH = Path("/kaggle/input/datasets/mduy2911/legaluit/LegalIR/selected-contexts")
DENSE_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/modeluit/bge-m3-kaggle")
RERANKER_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/modeluit/bge-reranker-v2-m3-kaggle")
BM25_WHEEL_PATH = Path("/kaggle/input/datasets/mduy2911/offline-packages/bm25s-0.3.11-py3-none-any.whl")

DENSE_MODEL_NAME = "BAAI/bge-m3"
DENSE_DECLARED_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANKER_DECLARED_REVISION = "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"
MODEL_PARAMETER_LIMIT = 4_000_000_000

EXPECTED_PUBLIC_SAMPLES = 1_000
EXPECTED_DOCUMENTS = 8_532
EXPECTED_CHUNKS = 199_816
CHUNK_SIZE, CHUNK_OVERLAP, CHUNK_STEP = 2_000, 200, 1_800
TOP_K_CHUNKS = 2_000
DENSE_ALPHA, BM25_ALPHA = 0.50, 0.50
CANDIDATE_DEPTH, SUPPORT_POOL_SIZE = 100, 8
DENSE_AGGREGATION_CHUNKS = CE_SELECTED_CHUNKS = 2
DENSE_MAX_LENGTH = RERANKER_MAX_SEQUENCE_LENGTH = 8_192
RRF_CONSTANT, DENSE_RANK_LAMBDA = 60, 0.25
FINAL_K = 5

BM25_VERSION = "0.3.11"
BM25_METHOD, BM25_K1, BM25_B = "lucene", 1.5, 0.75
BM25_TOKEN_PATTERN = r"(?u)\w+"

CORPUS_BATCH_SIZE, QUERY_BATCH_SIZE, RERANKER_BATCH_SIZE = 256, 64, 128
OUTPUT_PATH = Path("/kaggle/working/submission.json")
METADATA_PATH = Path("/kaggle/working/legalir_public_inference_metadata.json")


In [ ]:
# Enforce offline/local-only execution before imports and model loading.
import os
import subprocess
import sys

os.environ.update({
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "CUDA_VISIBLE_DEVICES": "0",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})

for path, description, must_be_directory in (
    (PUBLIC_LEGALIR_PATH, "official public LegalIR input", False),
    (CORPUS_PATH, "LegalIR corpus directory", True),
    (DENSE_MODEL_PATH, "complete local BGE-M3 snapshot", True),
    (RERANKER_MODEL_PATH, "complete local BGE reranker snapshot", True),
    (BM25_WHEEL_PATH, "bm25s 0.3.11 wheel", False),
):
    exists = path.is_dir() if must_be_directory else path.is_file()
    if not exists:
        raise FileNotFoundError(f"Attach the {description} at: {path}")

for name, value in (
    ("CORPUS_BATCH_SIZE", CORPUS_BATCH_SIZE),
    ("QUERY_BATCH_SIZE", QUERY_BATCH_SIZE),
    ("RERANKER_BATCH_SIZE", RERANKER_BATCH_SIZE),
):
    if not isinstance(value, int) or isinstance(value, bool) or value <= 0:
        raise ValueError(f"{name} must be a positive integer")

assert CHUNK_SIZE == 2_000 and CHUNK_OVERLAP == 200 and CHUNK_STEP == 1_800
assert TOP_K_CHUNKS == 2_000
assert DENSE_ALPHA == 0.50 and BM25_ALPHA == 0.50
assert CANDIDATE_DEPTH == 100 and SUPPORT_POOL_SIZE == 8
assert DENSE_AGGREGATION_CHUNKS == 2 and CE_SELECTED_CHUNKS == 2
assert RRF_CONSTANT == 60 and DENSE_RANK_LAMBDA == 0.25
assert FINAL_K == 5
assert OUTPUT_PATH == Path("/kaggle/working/submission.json")
assert OUTPUT_PATH != METADATA_PATH

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-index", str(BM25_WHEEL_PATH)
])


In [ ]:
# Standalone public LegalIR pipeline. No repository runtime is imported.
import gc
import json
import re
from collections import Counter, defaultdict
from math import isfinite
from time import perf_counter

import bm25s
import numpy as np
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer


if bm25s.__version__ != BM25_VERSION:
    raise RuntimeError(f"expected bm25s=={BM25_VERSION}, got {bm25s.__version__}")


def reject_duplicate_object_keys(pairs):
    value = {}
    for key, item in pairs:
        if key in value:
            raise ValueError(f"duplicate JSON object key: {key!r}")
        value[key] = item
    return value


def read_json_strict(path: Path):
    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream, object_pairs_hook=reject_duplicate_object_keys)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc


def load_public_legalir(path: Path) -> dict:
    value = read_json_strict(path)
    if not isinstance(value, dict):
        raise ValueError(f"{path}: expected a top-level object keyed by sample ID")
    if len(value) != EXPECTED_PUBLIC_SAMPLES:
        raise ValueError(f"expected {EXPECTED_PUBLIC_SAMPLES} public samples, got {len(value)}")
    for sample_id, sample in value.items():
        if not isinstance(sample_id, str):
            raise TypeError("public sample IDs must be JSON strings")
        if not isinstance(sample, dict) or set(sample) != {"question", "answer"}:
            raise ValueError(f"sample {sample_id!r}: expected exactly question and answer fields")
        if not isinstance(sample["question"], str):
            raise TypeError(f"sample {sample_id!r}: question must be a string")
        if sample["answer"] is not None:
            raise ValueError(f"sample {sample_id!r}: public answer must be explicit null")
    return value


def canonical_corpus_id(raw_id, source: Path) -> str:
    if not isinstance(raw_id, (str, int)) or isinstance(raw_id, bool):
        raise TypeError(f"{source}: corpus document ID must be a string or integer")
    if isinstance(raw_id, str) and not raw_id:
        raise ValueError(f"{source}: corpus document ID must not be empty")
    return str(raw_id)


def load_corpus(path: Path) -> list[dict]:
    paths = sorted(item for item in path.rglob("*") if item.is_file() and item.suffix.lower() == ".json")
    if not paths:
        raise ValueError(f"{path}: corpus directory contains no JSON files")
    documents, seen_ids = [], set()
    for json_path in paths:
        value = read_json_strict(json_path)
        values = value if isinstance(value, list) else [value]
        if not all(isinstance(document, dict) for document in values):
            raise ValueError(f"{json_path}: expected document object(s)")
        for document in values:
            document_id = canonical_corpus_id(document.get("id"), json_path)
            if document_id in seen_ids:
                raise ValueError(f"duplicate corpus document ID: {document_id!r}")
            passage = document.get("passage")
            if not isinstance(passage, str):
                raise TypeError(f"document {document_id!r}: passage must be a string")
            seen_ids.add(document_id)
            documents.append({"document_id": document_id, "passage": passage})
    if len(documents) != EXPECTED_DOCUMENTS:
        raise RuntimeError(f"expected {EXPECTED_DOCUMENTS} documents, got {len(documents)}")
    return documents


def distribution(values: list[int]) -> dict:
    array = np.asarray(values, dtype=np.int64)
    if array.size == 0:
        return {"min": None, "median": None, "p95": None, "max": None}
    return {
        "min": int(array.min()),
        "median": float(np.median(array)),
        "p95": float(np.percentile(array, 95)),
        "max": int(array.max()),
    }


def validate_chunk_provenance(documents: list[dict], chunks: list[dict]) -> dict:
    source_by_id = {document["document_id"]: document["passage"] for document in documents}
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]
    if len(chunk_ids) != len(set(chunk_ids)):
        raise RuntimeError("chunk IDs must be unique")
    intervals = defaultdict(list)
    for chunk in chunks:
        document_id = chunk["document_id"]
        source = source_by_id.get(document_id)
        start, end = chunk["char_start"], chunk["char_end"]
        if source is None or not (0 <= start < end <= len(source)) or chunk["text"] != source[start:end]:
            raise RuntimeError(f"chunk {chunk['chunk_id']!r}: exact provenance failed")
        intervals[document_id].append((start, end))
    for document_id, source in source_by_id.items():
        if not source:
            continue
        ordered = sorted(intervals[document_id])
        if not ordered or ordered[0][0] != 0:
            raise RuntimeError(f"document {document_id!r}: coverage does not start at zero")
        covered_end = 0
        for start, end in ordered:
            if start > covered_end:
                raise RuntimeError(f"document {document_id!r}: chunk coverage gap")
            covered_end = max(covered_end, end)
        if covered_end != len(source):
            raise RuntimeError(f"document {document_id!r}: incomplete source coverage")
    return {
        "exact_source_slices": True,
        "unique_chunk_ids": True,
        "complete_non_empty_source_coverage": True,
    }


def fixed_window_chunks(documents: list[dict], chunk_size: int, overlap: int) -> tuple[list[dict], dict]:
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("invalid fixed-window parameters")
    step = chunk_size - overlap
    if step != CHUNK_STEP:
        raise RuntimeError("fixed-window step mismatch")
    chunks, counts = [], []
    for document in documents:
        document_id, source = document["document_id"], document["passage"]
        before = len(chunks)
        for chunk_index, start in enumerate(range(0, len(source), step)):
            end = min(start + chunk_size, len(source))
            chunks.append({
                "chunk_id": f"{document_id}:{chunk_index}",
                "document_id": document_id,
                "chunk_index": chunk_index,
                "char_start": start,
                "char_end": end,
                "text": source[start:end],
            })
            if end == len(source):
                break
        counts.append(len(chunks) - before)
    provenance = validate_chunk_provenance(documents, chunks)
    return chunks, {
        "number_of_chunks": len(chunks),
        "chunk_size": chunk_size,
        "overlap": overlap,
        "step": step,
        "chunk_length_characters": distribution([len(chunk["text"]) for chunk in chunks]),
        "chunks_per_document": distribution(counts),
        "provenance": provenance,
    }


def model_metadata(model, model_name: str, declared_revision: str, path: Path) -> dict:
    if not re.fullmatch(r"[0-9a-f]{40}", declared_revision):
        raise RuntimeError(f"{model_name}: declared revision must be a resolved 40-hex commit SHA")
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    eligible = parameter_count < MODEL_PARAMETER_LIMIT
    print(
        "Model eligibility:\n"
        f"model = {model_name}\n"
        f"parameter_count = {parameter_count}\n"
        f"competition_limit = < {MODEL_PARAMETER_LIMIT:,}\n"
        f"eligible = {str(eligible).lower()}"
    )
    if not eligible:
        raise RuntimeError(f"{model_name} is ineligible: parameter_count >= 4,000,000,000")
    value = getattr(model.config, "_commit_hash", None)
    config_hash = value.strip() if isinstance(value, str) and value.strip() else None
    if config_hash is not None and config_hash != declared_revision:
        raise RuntimeError(f"{model_name} config revision {config_hash!r} != declared {declared_revision!r}")
    return {
        "model_repository": model_name,
        "declared_revision": declared_revision,
        "resolved_snapshot": config_hash or declared_revision,
        "config_commit_hash": config_hash,
        "revision_status": "verified-from-config" if config_hash else "declared-offline-snapshot",
        "actual_parameter_count": int(parameter_count),
        "competition_parameter_limit_exclusive": MODEL_PARAMETER_LIMIT,
        "eligible_under_4b_rule": eligible,
        "local_path": str(path),
        "local_files_only": True,
        "trust_remote_code": False,
    }


def load_dense_model() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle CUDA accelerator")
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL_PATH, local_files_only=True)
    model = AutoModel.from_pretrained(DENSE_MODEL_PATH, dtype=torch.float16, local_files_only=True)
    if int(getattr(model.config, "max_position_embeddings", 0)) < DENSE_MAX_LENGTH:
        raise RuntimeError("local dense model does not support max_length=8192")
    metadata = model_metadata(model, DENSE_MODEL_NAME, DENSE_DECLARED_REVISION, DENSE_MODEL_PATH)
    model.to("cuda").eval()
    return {"tokenizer": tokenizer, "model": model, "metadata": metadata, "load_seconds": perf_counter() - started}


def load_reranker() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle CUDA accelerator")
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_PATH, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL_PATH, dtype=torch.float16, local_files_only=True
    )
    if int(getattr(model.config, "max_position_embeddings", 0)) < RERANKER_MAX_SEQUENCE_LENGTH:
        raise RuntimeError("local reranker does not support max_sequence_length=8192")
    metadata = model_metadata(model, RERANKER_MODEL_NAME, RERANKER_DECLARED_REVISION, RERANKER_MODEL_PATH)
    model.to("cuda").eval()
    return {"tokenizer": tokenizer, "model": model, "metadata": metadata, "load_seconds": perf_counter() - started}


def encode_normalized_cls(model_bundle: dict, texts: list[str], batch_size: int) -> dict:
    embeddings, started = [], perf_counter()
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        inputs = model_bundle["tokenizer"](
            batch, padding=True, truncation=True, max_length=DENSE_MAX_LENGTH, return_tensors="pt"
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        with torch.no_grad():
            output = model_bundle["model"](**inputs, return_dict=True)
            embedding = F.normalize(output.last_hidden_state[:, 0], p=2, dim=1)
        if embedding.ndim != 2 or not torch.isfinite(embedding).all():
            raise RuntimeError("dense encoder returned invalid CLS embeddings")
        embeddings.append(embedding.cpu())
    encoded = torch.cat(embeddings, dim=0)
    if encoded.shape[0] != len(texts):
        raise RuntimeError("dense embedding count mismatch")
    return {"embeddings": encoded, "seconds": perf_counter() - started}


def retrieve_dense_hits(query_embeddings: torch.Tensor, corpus_embeddings: torch.Tensor, chunks: list[dict], sample_ids: list[str]) -> dict:
    if query_embeddings.shape[0] != len(sample_ids) or corpus_embeddings.shape[0] != len(chunks):
        raise ValueError("embedding count mismatch")
    if len(chunks) < TOP_K_CHUNKS:
        raise ValueError("corpus has fewer chunks than requested retrieval depth")
    started = perf_counter()
    corpus_gpu, hits = corpus_embeddings.to("cuda"), {}
    for start in range(0, len(sample_ids), QUERY_BATCH_SIZE):
        batch_ids = sample_ids[start:start + QUERY_BATCH_SIZE]
        query_gpu = query_embeddings[start:start + len(batch_ids)].to("cuda")
        similarities = query_gpu @ corpus_gpu.T
        if not torch.isfinite(similarities).all():
            raise RuntimeError("dense similarity contains non-finite values")
        scores, indices = torch.topk(similarities, k=TOP_K_CHUNKS, dim=1, sorted=True)
        for row, sample_id in enumerate(batch_ids):
            ordered = list(zip(scores[row].float().cpu().tolist(), indices[row].cpu().tolist()))
            ordered.sort(key=lambda item: (-item[0], item[1]))
            hits[sample_id] = [
                {
                    "chunk_index": int(chunk_index),
                    "document_id": chunks[int(chunk_index)]["document_id"],
                    "score": float(score),
                    "chunk_rank": rank,
                }
                for rank, (score, chunk_index) in enumerate(ordered, start=1)
            ]
    torch.cuda.synchronize()
    seconds = perf_counter() - started
    del corpus_gpu
    torch.cuda.empty_cache()
    return {"hits": hits, "seconds": seconds}


def retrieve_bm25_hits(chunks: list[dict], samples: dict) -> dict:
    tokenized = bm25s.tokenize(
        [chunk["text"] for chunk in chunks], lower=True, token_pattern=BM25_TOKEN_PATTERN,
        stopwords=[], stemmer=None, return_ids=True, show_progress=False,
    )
    index = bm25s.BM25(k1=BM25_K1, b=BM25_B, method=BM25_METHOD)
    index.index(tokenized, show_progress=False)
    output, started = {}, perf_counter()
    items = list(samples.items())
    for start in range(0, len(items), QUERY_BATCH_SIZE):
        batch = items[start:start + QUERY_BATCH_SIZE]
        queries = [re.findall(BM25_TOKEN_PATTERN, sample["question"].lower()) for _, sample in batch]
        result = index.retrieve(queries, k=TOP_K_CHUNKS, sorted=True, return_as="tuple", show_progress=False)
        for (sample_id, _), indices, scores in zip(batch, result.documents, result.scores):
            ordered = sorted(
                ((float(score), int(idx)) for idx, score in zip(indices, scores)),
                key=lambda item: (-item[0], item[1]),
            )
            if len(ordered) != TOP_K_CHUNKS or not all(isfinite(score) for score, _ in ordered):
                raise RuntimeError("BM25 returned an invalid top-2000 chunk ranking")
            output[sample_id] = [
                {"chunk_index": idx, "document_id": chunks[idx]["document_id"], "score": score, "chunk_rank": rank}
                for rank, (score, idx) in enumerate(ordered, 1)
            ]
    return {"hits": output, "seconds": perf_counter() - started}


def weighted_score_fuse(left: dict, right: dict, alpha: float) -> dict:
    if alpha != DENSE_ALPHA or 1.0 - alpha != BM25_ALPHA:
        raise ValueError("this frozen notebook permits only dense=0.50 / BM25=0.50")
    if set(left) != set(right):
        raise RuntimeError("dense and BM25 query IDs differ")
    fused = {}
    for sample_id in left:
        scores, best_rank = defaultdict(float), {}
        for branch, weight in ((left[sample_id], alpha), (right[sample_id], 1.0 - alpha)):
            values = [hit["score"] for hit in branch]
            low, high = min(values), max(values)
            denominator = high - low
            for hit in branch:
                normalized = 0.0 if denominator <= 1e-12 else (hit["score"] - low) / denominator
                chunk_index = hit["chunk_index"]
                scores[chunk_index] += weight * normalized
                best_rank[chunk_index] = min(best_rank.get(chunk_index, TOP_K_CHUNKS + 1), hit["chunk_rank"])
        ordered = sorted(scores, key=lambda index: (-scores[index], best_rank[index], index))[:TOP_K_CHUNKS]
        fused[sample_id] = [
            {"chunk_index": index, "document_id": chunks[index]["document_id"], "score": float(scores[index]), "chunk_rank": rank}
            for rank, index in enumerate(ordered, 1)
        ]
    return fused


def aggregate_top2(scores: list[float]) -> float:
    return sum(sorted(scores, reverse=True)[:DENSE_AGGREGATION_CHUNKS])


def candidates_from_hits(hits_by_query: dict, depth: int, support_limit: int = SUPPORT_POOL_SIZE, require_exact_depth: bool = True) -> dict:
    output = {}
    for sample_id, hits in hits_by_query.items():
        grouped = defaultdict(list)
        for hit in hits:
            if not isfinite(hit["score"]):
                raise RuntimeError("non-finite retrieval hit")
            grouped[hit["document_id"]].append(hit)
        documents = []
        for document_id, document_hits in grouped.items():
            ordered = sorted(document_hits, key=lambda item: (-item["score"], item["chunk_rank"], item["chunk_index"]))
            documents.append({
                "document_id": document_id,
                "document_score": aggregate_top2([item["score"] for item in ordered]),
                "best_chunk_rank": ordered[0]["chunk_rank"],
                "available_global_hits": len(ordered),
                "supporting_chunk_indices": [item["chunk_index"] for item in ordered[:support_limit]],
            })
        documents.sort(key=lambda item: (-item["document_score"], item["best_chunk_rank"], item["document_id"]))
        selected = documents[:depth]
        if not selected:
            raise RuntimeError(f"sample {sample_id!r}: retrieval produced no candidates")
        if require_exact_depth and len(selected) != depth:
            raise RuntimeError(f"sample {sample_id!r}: expected {depth} candidates")
        for rank, document in enumerate(selected, start=1):
            document["original_retrieval_rank"] = rank
        ids = [document["document_id"] for document in selected]
        if len(ids) != len(set(ids)):
            raise RuntimeError("candidate ranking contains duplicate document IDs")
        output[sample_id] = selected
    return output


def support_map_from_candidates(candidates: dict) -> dict:
    return {
        sample_id: {document["document_id"]: list(document["supporting_chunk_indices"]) for document in documents}
        for sample_id, documents in candidates.items()
    }


def make_ce_records(samples: dict, candidates: dict, supports: dict, chunks: list[dict]) -> list[tuple]:
    records = []
    for sample_id, sample in samples.items():
        for document in candidates[sample_id]:
            document_id = document["document_id"]
            indices = supports[sample_id][document_id]
            if not 1 <= len(indices) <= SUPPORT_POOL_SIZE:
                raise RuntimeError("support pool must contain one to eight chunks")
            for chunk_index in indices:
                key = ("ce", sample_id, document_id, chunk_index)
                records.append((key, sample["question"], chunks[chunk_index]["text"]))
    return records


def score_pair_records(reranker: dict, records: list[tuple[tuple, str, str]]) -> dict:
    cache, forward_seconds, started = {}, 0.0, perf_counter()
    for start in range(0, len(records), RERANKER_BATCH_SIZE):
        batch = records[start:start + RERANKER_BATCH_SIZE]
        inputs = reranker["tokenizer"](
            [record[1] for record in batch], [record[2] for record in batch],
            padding=True, truncation="only_second", max_length=RERANKER_MAX_SEQUENCE_LENGTH,
            return_tensors="pt",
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        torch.cuda.synchronize()
        forward_started = perf_counter()
        with torch.no_grad():
            logits = reranker["model"](**inputs, return_dict=True).logits.view(-1).float()
        torch.cuda.synchronize()
        forward_seconds += perf_counter() - forward_started
        scores = logits.cpu().tolist()
        if len(scores) != len(batch) or not all(isfinite(score) for score in scores):
            raise RuntimeError("reranker returned invalid scores")
        for record, score in zip(batch, scores):
            if record[0] in cache:
                raise RuntimeError("duplicate CE pair key")
            cache[record[0]] = float(score)
    return {
        "cache": cache,
        "diagnostics": {
            "pairs": len(records),
            "forward_seconds": forward_seconds,
            "total_seconds": perf_counter() - started,
        },
    }


def derive_ce_ranking(samples: dict, candidates: dict, supports: dict, score_cache: dict) -> dict:
    rankings = {}
    for sample_id in samples:
        scored_documents = []
        for original_rank, document in enumerate(candidates[sample_id], start=1):
            document_id = document["document_id"]
            scored_chunks = []
            for support_position, chunk_index in enumerate(supports[sample_id][document_id]):
                key = ("ce", sample_id, document_id, chunk_index)
                if key not in score_cache:
                    raise RuntimeError("missing CE score; no imputation is allowed")
                scored_chunks.append((score_cache[key], support_position, chunk_index))
            scored_chunks.sort(key=lambda item: (-item[0], item[1], item[2]))
            document_score = sum(item[0] for item in scored_chunks[:CE_SELECTED_CHUNKS])
            if not isfinite(document_score):
                raise RuntimeError("non-finite CE document score")
            scored_documents.append((document_id, document_score, original_rank))
        scored_documents.sort(key=lambda item: (-item[1], item[2], item[0]))
        ranking = [item[0] for item in scored_documents]
        expected = [document["document_id"] for document in candidates[sample_id]]
        if len(ranking) != len(set(ranking)) or set(ranking) != set(expected):
            raise RuntimeError("CE reranking changed the candidate set")
        rankings[sample_id] = ranking
    return rankings


def final_rank_fusion(ce_rankings: dict, dense_full_rank: dict) -> dict:
    fused, missing_dense_rank = {}, Counter()
    for sample_id, documents_ranked in ce_rankings.items():
        ce_rank = {document_id: rank for rank, document_id in enumerate(documents_ranked, 1)}
        fallback_rank = len(dense_full_rank[sample_id]) + 1
        rows = []
        for document_id, rank in ce_rank.items():
            dense_rank = dense_full_rank[sample_id].get(document_id, fallback_rank)
            missing_dense_rank["hybrid_0.50"] += document_id not in dense_full_rank[sample_id]
            score = 1.0 / (RRF_CONSTANT + rank) + DENSE_RANK_LAMBDA / (RRF_CONSTANT + dense_rank)
            rows.append((document_id, score, rank, dense_rank))
        rows.sort(key=lambda item: (-item[1], item[2], item[3], item[0]))
        fused[sample_id] = [item[0] for item in rows]
    return {"rankings": fused, "missing_dense_rank_count": int(missing_dense_rank["hybrid_0.50"])}


def build_predictions(public_samples: dict, rankings: dict) -> dict:
    predictions = {}
    for sample_id in public_samples:
        ranking = rankings.get(sample_id)
        if ranking is None:
            raise KeyError(f"sample {sample_id!r}: missing final ranking")
        answer = ranking[:FINAL_K]
        if not 1 <= len(answer) <= FINAL_K:
            raise ValueError(f"sample {sample_id!r}: answer must contain between one and five IDs")
        if len(ranking) >= FINAL_K and len(answer) != FINAL_K:
            raise ValueError(f"sample {sample_id!r}: five IDs required when available")
        if len(answer) != len(set(answer)):
            raise ValueError(f"sample {sample_id!r}: duplicate document IDs")
        if not all(isinstance(document_id, str) for document_id in answer):
            raise TypeError(f"sample {sample_id!r}: document IDs must be strings")
        predictions[sample_id] = {"answer": answer}
    return predictions


def validate_predictions(predictions: dict, public_samples: dict, corpus_ids: set[str]) -> dict:
    input_ids, prediction_ids = set(public_samples), set(predictions)
    missing_ids, unexpected_ids = input_ids - prediction_ids, prediction_ids - input_ids
    answer_lengths, duplicate_answer_count = Counter(), 0
    if len(public_samples) != EXPECTED_PUBLIC_SAMPLES or len(predictions) != EXPECTED_PUBLIC_SAMPLES:
        raise ValueError("public input/prediction sample count mismatch")
    if missing_ids or unexpected_ids:
        raise ValueError(f"prediction IDs mismatch: missing={len(missing_ids)}, unexpected={len(unexpected_ids)}")
    for sample_id, value in predictions.items():
        if not isinstance(value, dict) or set(value) != {"answer"}:
            raise ValueError(f"sample {sample_id!r}: output must contain answer only")
        answer = value["answer"]
        if not isinstance(answer, list) or not 1 <= len(answer) <= FINAL_K:
            raise TypeError(f"sample {sample_id!r}: answer must be list[str] with length 1..5")
        answer_lengths[len(answer)] += 1
        if len(answer) != len(set(answer)):
            duplicate_answer_count += 1
        if not all(isinstance(document_id, str) for document_id in answer):
            raise TypeError(f"sample {sample_id!r}: every document ID must be a string")
        if any(document_id not in corpus_ids for document_id in answer):
            raise ValueError(f"sample {sample_id!r}: answer contains an ID absent from corpus")
    if duplicate_answer_count:
        raise ValueError(f"duplicate document IDs found in {duplicate_answer_count} answers")
    if answer_lengths.get(FINAL_K, 0) != EXPECTED_PUBLIC_SAMPLES:
        raise RuntimeError("expected five document IDs for every public query with candidate100")
    return {
        "input_samples": len(public_samples),
        "prediction_samples": len(predictions),
        "missing_id_count": len(missing_ids),
        "unexpected_id_count": len(unexpected_ids),
        "expected_ids_exact_match": True,
        "answer_type": "list[str]",
        "answer_length_distribution": {str(length): answer_lengths[length] for length in sorted(answer_lengths)},
        "answer_length_between_1_and_5": True,
        "five_when_available": True,
        "duplicate_answer_count": duplicate_answer_count,
        "all_document_ids_strings": True,
        "all_document_ids_in_corpus": True,
    }


In [ ]:
# Run once on offline Kaggle. This is inference only and writes no evaluation result.
if not torch.cuda.is_available():
    raise RuntimeError("This inference notebook requires a Kaggle CUDA accelerator")

run_started = perf_counter()
torch.cuda.reset_peak_memory_stats()
public_samples = load_public_legalir(PUBLIC_LEGALIR_PATH)
documents = load_corpus(CORPUS_PATH)
chunks, chunking = fixed_window_chunks(documents, CHUNK_SIZE, CHUNK_OVERLAP)
if len(documents) != EXPECTED_DOCUMENTS or len(chunks) != EXPECTED_CHUNKS:
    raise RuntimeError(
        f"expected {EXPECTED_DOCUMENTS:,} documents / {EXPECTED_CHUNKS:,} chunks, "
        f"got {len(documents):,} / {len(chunks):,}"
    )
corpus_ids = {document["document_id"] for document in documents}
if len(corpus_ids) != EXPECTED_DOCUMENTS:
    raise RuntimeError("corpus ID cardinality mismatch")

dense = load_dense_model()
dense_metadata, dense_load_seconds = dict(dense["metadata"]), dense["load_seconds"]
corpus_encoding = encode_normalized_cls(dense, [chunk["text"] for chunk in chunks], CORPUS_BATCH_SIZE)
query_encoding = encode_normalized_cls(
    dense, [sample["question"] for sample in public_samples.values()], QUERY_BATCH_SIZE
)
dense_retrieval = retrieve_dense_hits(
    query_encoding["embeddings"], corpus_encoding["embeddings"], chunks, list(public_samples)
)
bm25_retrieval = retrieve_bm25_hits(chunks, public_samples)
hybrid_hits = weighted_score_fuse(dense_retrieval["hits"], bm25_retrieval["hits"], DENSE_ALPHA)
hybrid_candidates = candidates_from_hits(hybrid_hits, CANDIDATE_DEPTH)

# Preserve the validated DEV missing-rank policy exactly: rank after the BGE top-2000-derived document pool.
dense_full = candidates_from_hits(
    dense_retrieval["hits"], EXPECTED_DOCUMENTS, require_exact_depth=False
)
dense_full_rank = {
    sample_id: {document["document_id"]: rank for rank, document in enumerate(documents_ranked, 1)}
    for sample_id, documents_ranked in dense_full.items()
}

dense["model"].to("cpu")
del dense, corpus_encoding["embeddings"], query_encoding["embeddings"]
gc.collect()
torch.cuda.empty_cache()

reranker = load_reranker()
reranker_metadata, reranker_load_seconds = dict(reranker["metadata"]), reranker["load_seconds"]
supports = support_map_from_candidates(hybrid_candidates)
records = make_ce_records(public_samples, hybrid_candidates, supports, chunks)
scored = score_pair_records(reranker, records)
ce_rankings = derive_ce_ranking(public_samples, hybrid_candidates, supports, scored["cache"])
final_fusion = final_rank_fusion(ce_rankings, dense_full_rank)
predictions = build_predictions(public_samples, final_fusion["rankings"])
validation = validate_predictions(predictions, public_samples, corpus_ids)


In [ ]:
# Write the scorer-facing artifact and a prediction-free audit sidecar.
metadata = {
    "task": "LegalIR public inference",
    "pipeline_description": (
        "fixed 2000/200 source windows -> BGE-M3 normalized CLS top2000 + BM25 Lucene top2000 "
        "-> per-query min-max 0.50/0.50 chunk-score fusion -> fused top2000 -> document sum-top2 "
        "-> candidate100 -> m8 hybrid supports -> BGE reranker independent chunk scoring "
        "-> CE select top2/sum top2 -> CE RR + 0.25 * original BGE-dense RR (k=60) -> top5"
    ),
    "previous_public_reference": {
        "score": 0.8935,
        "method": "BGE-M3 dense top2000 -> sum-top2 -> candidate100 -> m8 -> CE top2/sum-top2 -> top5",
        "status": "retained until this new submission receives an actual public score",
    },
    "selected_dev_evidence": {
        "baseline": {"precision": 0.19150579150579153, "recall": 0.8973616473616474, "mrr": 0.7622471799366903},
        "selected_candidate": {"precision": 0.19420849420849426, "recall": 0.9105534105534104, "mrr": 0.7710374420013412},
        "delta": {"precision": 0.00270270270270273, "recall": 0.0131917631917630, "mrr": 0.0087902620646509},
        "paired_bootstrap_95_percent_ci": {
            "precision": [0.0009652509652509653, 0.004633204633204633],
            "recall": [0.005308880308880309, 0.02171814671814672],
        },
        "paired_bootstrap_fraction_delta_gt_0": {"precision": 0.9979, "recall": 0.9996},
    },
    "input_path": str(PUBLIC_LEGALIR_PATH),
    "output_path": str(OUTPUT_PATH),
    "metadata_path": str(METADATA_PATH),
    "models": {"dense": dense_metadata, "reranker": reranker_metadata},
    "competition_model_parameter_limit_exclusive": MODEL_PARAMETER_LIMIT,
    "all_neural_models_eligible_under_4b_rule": (
        dense_metadata["eligible_under_4b_rule"] and reranker_metadata["eligible_under_4b_rule"]
    ),
    "chunking": {
        **chunking,
        "representation": "source-preserving fixed character windows",
        "fields": ["chunk_id", "document_id", "chunk_index", "char_start", "char_end", "text"],
        "title_enrichment": False,
        "article_aware": False,
        "neighbor_context": False,
        "full_document_context": False,
    },
    "dense_retrieval": {
        "embedding": "L2-normalized CLS last hidden state",
        "similarity": "dot product",
        "max_length": DENSE_MAX_LENGTH,
        "top_k_chunks": TOP_K_CHUNKS,
        "document_aggregation_for_final_dense_rank": "sum top-2 scores over original BGE top-2000 chunks",
    },
    "bm25": {
        "package": f"bm25s=={bm25s.__version__}",
        "method": BM25_METHOD,
        "k1": BM25_K1,
        "b": BM25_B,
        "lowercase": True,
        "token_pattern": "Unicode \\w+",
        "stemming": False,
        "stopwords": False,
        "top_k_chunks": TOP_K_CHUNKS,
    },
    "candidate_fusion": {
        "normalization": "per-query min-max; identical-score fallback=0; missing branch contribution=0",
        "dense_alpha": DENSE_ALPHA,
        "bm25_alpha": BM25_ALPHA,
        "fused_top_k_chunks": TOP_K_CHUNKS,
        "document_aggregation": "sum top-2 hybrid chunk scores",
        "candidate_depth": CANDIDATE_DEPTH,
        "support_pool": SUPPORT_POOL_SIZE,
        "support_source": "same fused top-2000 pool ordered by hybrid chunk score",
    },
    "cross_encoder": {
        "max_sequence_length": RERANKER_MAX_SEQUENCE_LENGTH,
        "pair_scope": "question and one hybrid support chunk scored independently",
        "aggregation": "select top-2 CE scores and sum top-2",
    },
    "final_fusion": {
        "formula": "1/(60+ce_rank) + 0.25/(60+dense_rank)",
        "rrf_constant": RRF_CONSTANT,
        "dense_lambda": DENSE_RANK_LAMBDA,
        "dense_rank_source": "original BGE-M3 dense sum-top2 document ranking from BGE top-2000 chunks",
        "missing_dense_rank_policy": "len(BGE top2000-derived document pool) + 1",
        "missing_dense_rank_count": final_fusion["missing_dense_rank_count"],
        "final_k": FINAL_K,
    },
    "counts": {
        "queries": len(public_samples),
        "corpus_documents": len(documents),
        "chunks": len(chunks),
    },
    "runtime": {
        "dense_model_load_seconds": dense_load_seconds,
        "corpus_encoding_seconds": corpus_encoding["seconds"],
        "query_encoding_seconds": query_encoding["seconds"],
        "dense_retrieval_seconds": dense_retrieval["seconds"],
        "bm25_retrieval_seconds": bm25_retrieval["seconds"],
        "reranker_load_seconds": reranker_load_seconds,
        "cross_encoder": scored["diagnostics"],
        "total_seconds": perf_counter() - run_started,
        "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated()),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
    },
    "submission_validation": validation,
}

if not metadata["all_neural_models_eligible_under_4b_rule"]:
    raise RuntimeError("refusing to write output because a neural model is not <4B")
if "predictions" in metadata or "answers" in metadata:
    raise RuntimeError("metadata must not contain predictions")

OUTPUT_PATH.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print(
    "Prediction file:\n"
    f"  {OUTPUT_PATH}\n\n"
    "Inference metadata (no predictions):\n"
    f"  {METADATA_PATH}\n\n"
    "For Codabench:\n"
    "  package ONLY submission.json"
)
